# Answers — Python / Pandas / NumPy Essentials

Check only after attempting the main notebook.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT=next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/"reference"/"dataset_loader.py").exists()),Path.cwd())
sys.path.insert(0,str(ROOT/"reference"))
from dataset_loader import load_checked_csv
URL="https://raw.githubusercontent.com/Rawanmalharbi/UCI-Bank-Marketing/main/bank-full.csv"
df=load_checked_csv("bank_full.csv",URL,expected_cols=["age","job","marital","education","balance","day","month","duration","campaign","pdays","previous","poutcome","y"],min_rows=45000,read_csv_kwargs={"sep":";"})

## A. Audit

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print("duplicates",df.duplicated().sum())
print(df.nunique().sort_values())
print(df["y"].value_counts(normalize=True))
display(df.describe())

## B. Filtering

In [ ]:
print(((df["age"]>50)&(df["balance"]<0)).sum())
print((df.loc[df["age"]>60,"y"]=="yes").mean())
display(df.loc[df["age"]<30].nlargest(20,"balance"))

## C/D. Groupby, transform, merge

In [ ]:
tmp=df.assign(target=(df["y"]=="yes").astype(int))
job_summary=(tmp.groupby("job").agg(n=("target","size"),mean_age=("age","mean"),median_balance=("balance","median"),subscription_rate=("target","mean")).reset_index())
display(job_summary)
tmp["job_mean_balance"]=tmp.groupby("job")["balance"].transform("mean")
tmp["balance_demeaned"]=tmp["balance"]-tmp["job_mean_balance"]
merged=df.merge(job_summary,on="job",how="left",validate="many_to_one")
print(len(df),len(merged))

## E. Pivot / melt

In [ ]:
tmp=df.assign(target=(df["y"]=="yes").astype(int))
p=pd.pivot_table(tmp,index="job",columns="marital",values="target",aggfunc="mean")
display(p)
long=p.reset_index().melt(id_vars="job",var_name="marital",value_name="subscription_rate")
display(long.head())

## F. Cyclic month

In [ ]:
month_map=dict(zip(["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"],range(1,13)))
m=df["month"].map(month_map)
month_sin=np.sin(2*np.pi*m/12); month_cos=np.cos(2*np.pi*m/12)
display(pd.DataFrame({"month":df["month"].head(),"sin":month_sin.head(),"cos":month_cos.head()}))

## G. Unknown counts

In [ ]:
obj=df.select_dtypes(include="object").columns
print(pd.Series({c:(df[c]=="unknown").sum() for c in obj}).sort_values(ascending=False))

## H. NumPy

In [ ]:
X=df[["age","balance","campaign","previous"]].to_numpy(float)
print(X.shape); print(X[:,0][:5]); print(X.mean(axis=0)); print(X.mean(axis=1)[:5])
mask=X[:,0]>50; print(X[mask].shape)
mu=X.mean(axis=0); sd=X.std(axis=0); Z=(X-mu)/sd
print(Z.mean(axis=0)); print(Z.std(axis=0)); print(np.quantile(X,[.01,.5,.99],axis=0))

## I. Plotting

In [ ]:
tmp=df.assign(target=(df["y"]=="yes").astype(int))
fig,ax=plt.subplots(figsize=(5,4)); tmp["target"].value_counts().sort_index().plot(kind="bar",ax=ax); ax.set_title("Target"); plt.show()
fig,ax=plt.subplots(figsize=(7,4)); ax.hist(df["age"],bins=30); ax.set_title("Age"); plt.show()
lo,hi=df["balance"].quantile([.01,.99]); fig,ax=plt.subplots(figsize=(7,4)); ax.hist(df["balance"].clip(lo,hi),bins=40); ax.set_title("Balance clipped 1/99%"); plt.show()
tmp["age_bucket"]=pd.cut(tmp["age"],bins=[0,30,40,50,60,100]); tmp.groupby("age_bucket",observed=True)["target"].mean().plot(kind="bar"); plt.ylabel("Subscription rate"); plt.tight_layout(); plt.show()
corr=tmp.select_dtypes(include=np.number).corr(); fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(corr,vmin=-1,vmax=1,aspect="auto"); fig.colorbar(im,ax=ax); plt.show()